# Alternative 3b — GAT (Graph Attention Network)

**Objectif :** Comparer la variante attentionnelle du GNN : au lieu de l'adjacence normalisée fixe, chaque nœud apprend des poids d'attention sur ses voisins anatomiques.

Le GAT remplace l'agrégation fixe du GCN par une attention paire-à-paire
(GATv1, 4 têtes) restreinte aux mêmes arêtes anatomiques. Structure de pooling et tête
identiques au GCN pour une comparaison directe GCN vs GAT.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit graphe de 21 nœuds × 2 mains, features par nœud : (x, y, z).


In [2]:
dataset = load_dataset()
X_train = torch.from_numpy(dataset["train"]["img_landmarks"]).float()
y_train = torch.from_numpy(dataset["train"]["labels"])
X_val   = torch.from_numpy(dataset["val"]["img_landmarks"]).float()
y_val   = torch.from_numpy(dataset["val"]["labels"])
X_test  = torch.from_numpy(dataset["test"]["img_landmarks"]).float()
y_test  = torch.from_numpy(dataset["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 2, 21, 3) | val: (767, 2, 21, 3) | test: (767, 2, 21, 3)


## 2. Architecture

GAT : 2 couches d'attention (4 têtes × 16D), arêtes anatomiques uniquement,
BatchNorm + ReLU, pooling mean+max masqué par main, tête 128 → 57. ~53k paramètres.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = HandGAT(num_classes=57, hidden=64, num_layers=2, n_heads=4)
print(f"params: {count_params(model):,}")


params: 53,497


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 100 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=100, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "gat_curves.png",
                     title="Alternative 3b — GAT (Graph Attention Network)")
print("best val acc:", trainer.best_val_acc)


  ep 1/100 train 3.3296/0.1889 val 3.1288/0.2282 (4.2s) best 0.2282


  ep 2/100 train 2.7521/0.3369 val 2.6211/0.3546 (4.3s) best 0.3546


  ep 3/100 train 2.4742/0.4261 val 2.4893/0.4211 (3.4s) best 0.4211


  ep 4/100 train 2.2962/0.4851 val 2.2273/0.5085 (4.2s) best 0.5085


  ep 5/100 train 2.1742/0.5264 val 2.0716/0.5671 (4.2s) best 0.5671


...

  ep 32/100 train 1.5955/0.7302 val 1.7268/0.6584 (6.3s) best 0.6767


  ep 33/100 train 1.6011/0.7280 val 1.7660/0.6454 (4.9s) best 0.6767


  ep 34/100 train 1.5948/0.7315 val 1.7866/0.6323 (4.7s) best 0.6767


  ep 35/100 train 1.5754/0.7402 val 1.7818/0.6336 (3.6s) best 0.6767
  early stop @ 35 (best val acc 0.6767)


best val acc: 0.6766623207301173


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "gat_cm.png",
               title="Alternative 3b — GAT (Graph Attention Network) — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_gat.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_gat.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.6858
macro F1: 0.6666 | weighted F1: 0.6859
macro precision: 0.7327 | macro recall: 0.6698


     class  precision    recall        f1  support
10  mar7ba   0.000000  0.000000  0.000000        3
30  sbitar   0.000000  0.000000  0.000000        9
54   metro   0.000000  0.000000  0.000000        0
8    labes   0.111111  0.250000  0.153846        4
20  ta3raf   1.000000  0.166667  0.285714        6
17  siye7a   0.400000  0.222222  0.285714        9
48    sebt   0.285714  0.285714  0.285714        7
39   jadda   0.296296  0.500000  0.372093       16


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(raw), batch_size):
            x = torch.from_numpy(raw[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.6010 (drop=+0.0847)


  noise_0.02: acc=0.5838 (drop=+0.1020)


  noise_0.05: acc=0.4050 (drop=+0.2808)


  rotation_5: acc=0.6780 (drop=+0.0078)


  rotation_10: acc=0.6597 (drop=+0.0261)


  rotation_20: acc=0.6206 (drop=+0.0652)


  translation_0.05: acc=0.5953 (drop=+0.0905)
  translation_0.10: acc=0.5359 (drop=+0.1499)


  scale_0.90: acc=0.5932 (drop=+0.0926)
  scale_1.10: acc=0.7458 (drop=-0.0600)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "gat")
fp16 = ex.quantize_fp16(onnx_path, "gat")
int8 = ex.quantize_int8(onnx_path, "gat")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("gat", metrics, extra=extra)

torch_path = ex.save_torch(model, "gat")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.535 ms/sample


  fp32: size=0.22 MB latency=0.279 ms acc=0.6858
  fp16: size=0.12 MB latency=0.275 ms acc=0.6858


  int8: size=0.08 MB latency=0.266 ms acc=0.6845
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_gat.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\gat.pth (0.21 MB)


## 7. Conclusions

Le GAT apprend si certaines connexions anatomiques sont plus importantes que
d'autres pour discriminer les signes. Comparé au GCN, il ajoute de la flexibilité au prix de
légèrement plus de paramètres.
